# Sequence Optimization with GOOSE

This notebook demonstrates the functionality of the `SequenceOptimizer` from GOOSE, using various protein properties defined in the optimizer_properties module. We will:
- Initialize a SequenceOptimizer
- Add multiple properties (e.g., Hydrophobicity, FCR, NCPR)
- Set optimization parameters
- Run the optimization
- Analyze the optimized sequence


In [ ]:
# Import required modules
import goose
from goose.optimize import SequenceOptimizer
from sparrow import Protein


## Steps in optimization...
### Initialize SequenceOptimizer
Create an instance of SequenceOptimizer with a target sequence length. You can also specify other initialization parameters as needed.

### Add Protein Properties
Add one or more properties to optimize. Here, we add Hydrophobicity, FCR, and NCPR as examples. You can add as many properties as you like, each with its own target value and weight.

### Run the optimizer
optimizer.run() actually runs the optimizer.

In [ ]:

# Set the target sequence length
sequence_length = 100

# Initialize the optimizer
optimizer = SequenceOptimizer(target_length=sequence_length, 
                              verbose=True,
                              max_iterations=5000)

# Add properties to optimize
optimizer.add_property(goose.Hydrophobicity, target_value=2.5, weight=1.0)
optimizer.add_property(goose.FCR, target_value=0.35, weight=1.0, constraint_type='minimum')
optimizer.add_property(goose.NCPR, target_value=0.0, weight=1.0)

# Run the optimization
optimized_sequence = optimizer.run()



## Analyze Optimized Sequence

Evaluate the optimized sequence by calculating the values of the properties and comparing them to the targets.

In [ ]:
# Create a Protein object from the optimized sequence
final_protein = Protein(optimized_sequence)

# Calculate and print property values
hydro = final_protein.hydrophobicity
fcr = final_protein.FCR
ncpr = final_protein.NCPR


print(f"Final Hydrophobicity: {hydro:.3f} (Target: 2.5)")
print(f"Final FCR: {fcr:.3f} (Target: minimum 0.35)")
print(f"Final NCPR: {ncpr:.3f} (Target: 0.0)")


## Constraining amino-acid composition with `aa_fraction_ranges`

`SequenceOptimizer` can also enforce hard bounds on the per-residue and grouped amino-acid composition of every candidate it generates. This is useful when you want to satisfy property targets *while* keeping the sequence inside a particular biological niche -- for example, IDRs that are aromatic-poor.

`aa_fraction_ranges` accepts a dict whose keys are either single amino-acid letters or tuples/strings of grouped amino acids, and whose values are `(min_fraction, max_fraction)` bounds. Bounds are enforced on initial seeds, mutations, shuffles, and emergency diversity injection.

In [ ]:
# Same multi-property optimization, but cap aromatic content (W/F/Y) at 5%
# and require Pro content between 5% and 15%.
optimizer = SequenceOptimizer(
    target_length=100,
    verbose=True,
    max_iterations=2000,
    aa_fraction_ranges={
        ('W', 'F', 'Y'): (0.0, 0.05),  # aromatic-poor
        'P': (0.05, 0.15),             # proline-rich,
        'G': (0.0),                    # glycine-free
    },
)

optimizer.add_property(goose.FCR, target_value=0.30, weight=1.0)
optimizer.add_property(goose.NCPR, target_value=0.0, weight=1.0)
optimizer.add_property(goose.Hydrophobicity, target_value=2.5, weight=1.0)

constrained_seq = optimizer.run()

p = Protein(constrained_seq)
n = len(constrained_seq)
print(f"Length:        {n}")
print(f"FCR:           {p.FCR:.3f} (target 0.30)")
print(f"NCPR:          {p.NCPR:.3f} (target 0.00)")
print(f"Hydrophobicity:{p.hydrophobicity:.3f} (target 2.50)")
print(f"Aromatic frac: {sum(constrained_seq.count(a) for a in 'WFY')/n:.3f} (cap 0.05)")
print(f"Proline frac:  {constrained_seq.count('P')/n:.3f} (range 0.05-0.15)")
print(f"Glycine frac:  {constrained_seq.count('G')/n:.3f} (must be 0.0)")


## Warm-starting an optimization with `set_initial_sequence`

If you already have a candidate sequence (perhaps generated by a previous run or by `goose.create.sequence(...)`), you can hand it to the optimizer as the starting point with `set_initial_sequence()`. The optimizer will compute normalization factors against that sequence and begin local exploration around it instead of generating fresh random seeds. This is helpful when you want to retune one property without losing the work invested in the others.

In [ ]:
starting_seq = goose.create.sequence(100, hydropathy=3.2, NCPR=0.3)

# Re-optimize the previous sequence to add a stronger negative net charge
# while keeping hydrophobicity high.
optimizer = SequenceOptimizer(
    target_length=len(starting_seq),
    verbose=True,
    max_iterations=5000,
)

optimizer.add_property(goose.Hydrophobicity, target_value=2.6, weight=1.0)
optimizer.add_property(goose.NCPR, target_value=-0.20, weight=2.0)

# Warm-start from the previous result.
optimizer.set_initial_sequence(starting_seq)
warm_started_seq = optimizer.run()

before = Protein(starting_seq)
after = Protein(warm_started_seq)
print(f"NCPR  (before -> after): {before.NCPR:+.3f} -> {after.NCPR:+.3f} (target -0.20)")
print(f"Hydro (before -> after): {before.hydrophobicity:.3f} -> {after.hydrophobicity:.3f} (target 2.6)")
print(before.sequence)
print(after.sequence)